# pandas 之十一：数据清洗（下）——脏数据实战

---

上一篇讲增删改，这篇讲**真实的脏数据长什么样、怎么治**。

## 本篇覆盖的 10 类脏数据

| # | 问题 | 对应方法 |
|---|---|---|
| 1 | 缺失值 | `fillna` / `ffill` / `dropna` |
| 2 | 重复值 | `duplicated` / `drop_duplicates` |
| 3 | 按条件筛数据 | `query` |
| 4 | 按时间筛数据 | `.between` / `truncate` / `.dt` |
| 5 | 一列挤了多个字段 | `str.split(expand=True)` |
| 6 | 多个字段想拼一列 | `+` / `str.cat` |
| 7 | 内容要替换 | `str.replace` |
| 8 | 要从文字里抠数字 | `str.extract` |
| 9 | 一格里塞了一个列表 | `explode` |
| 10 | 多行想并成一行 | `groupby` + `agg(join)` |


In [ ]:
import pandas as pd
import numpy as np

print('pandas 版本：', pd.__version__)


---
# 一、缺失值处理

## 1.1 造一份带缺失值的数据


In [ ]:
df = pd.DataFrame({
    'A': [1, 2, np.nan, 4, 5],
    'B': [np.nan, 2, 3, np.nan, 5],
    'C': [1, 2, 3, 4, np.nan],
})
print(df)
print('\n每列缺失数：')
print(df.isna().sum())


## 1.2 填充策略总览（**按数据类型选**）

| 数据类型 | 推荐填充方式 | 理由 |
|---|---|---|
| 数值 · 近似正态 | **均值** `mean()` | 保持整体水平 |
| 数值 · **有极端值** | **中位数** `median()` | 不受极端值影响 ✅ |
| 分类 / 文字 | **众数** `mode()` | 最常见值最合理 |
| 时间序列 | `ffill()` / `bfill()` | 延续趋势 |
| 计数类 | `0` | "没发生"就是 0 |

> **经验**：**工资、房价、收入这类"有极端值"的数据，一定要用中位数**，
> 用均值会被少数超高值"带偏"。


In [ ]:
# 策略 1：数值列用均值
d1 = df.copy()
print('用各列均值填充（fillna(均值) 会自动按列对齐）：')
print(d1.fillna(d1.mean()).round(2))


In [ ]:
# 策略 2：用中位数（更抗极端值）
d2 = df.copy()
print('用各列中位数填充：')
print(d2.fillna(d2.median()))


In [ ]:
# 策略 3：众数填充（分类数据用）
d3 = pd.DataFrame({'城市': ['北京', '上海', '北京', np.nan, '北京', np.nan]})
print('原始：')
print(d3['城市'].tolist())

mode_city = d3['城市'].mode()[0]
print('\n众数 =', mode_city)
print('填充后：', d3['城市'].fillna(mode_city).tolist())


> **⚠️ `mode()[0]` 里的 `[0]` 千万别漏！**
> `mode()` 返回的是 Series（可能多个众数并列），
> 直接 `fillna(Series)` 会出问题。**取第一个：`mode()[0]`。**


In [ ]:
# 策略 4：前向填充 / 后向填充（时间序列首选）
d4 = pd.DataFrame({'日期': pd.date_range('2023-01-01', periods=6),
                   '价格': [100, np.nan, np.nan, 130, np.nan, 150]})
print('原始：')
print(d4)

print('\nffill 前向填充（沿用上一个有效值，适合"价格持续有效"的场景）：')
print(d4['价格'].ffill().tolist())

print('\nbfill 后向填充（用下一个值回填）：')
print(d4['价格'].bfill().tolist())


### ⚠️ 版本坑：`fillna(method='ffill')` 已经不能用了


In [ ]:
try:
    d4['价格'].fillna(method='ffill')
except TypeError as e:
    print("fillna(method='ffill') -> 报错：")
    print('  ', str(e)[:110])
    print('\n  <- pandas 3.0 已移除 method 参数，改用 .ffill() / .bfill()')


## 1.3 删除缺失值（`dropna`）

填充之外的另一条路。**但删之前要想清楚代价。**


In [ ]:
print('=== 原始 ===')
print(df)

print('\n=== dropna() 默认：只要有一个缺失就删这行 ===')
print(df.dropna())

print('\n=== how="all"：整行全缺才删 ===')
print(df.dropna(how='all'))

print('\n=== axis=1：删"有缺失的列" ===')
print(df.dropna(axis=1))


In [ ]:
print('=== thresh=2：保留至少 2 个非缺失的行 ===')
print(df.dropna(thresh=2))

print("\n=== subset=['A']：只看 A 列，A 缺了就删这行 ===")
print(df.dropna(subset=['A']))


**`dropna` 参数表**：

| 参数 | 含义 | 默认 |
|---|---|---|
| `axis` | 0=删行，1=删列 | 0 |
| `how` | `'any'` / `'all'` | `'any'` |
| `thresh` | 至少要几个非缺失 | None |
| `subset` | 只看指定列 | None |
| `inplace` | 是否改原表 | `False` |


In [ ]:
# 算一下 dropna 的"代价"，再决定要不要删
cost = len(df) - len(df.dropna())
print(f'原表 {len(df)} 行，dropna 后 {len(df.dropna())} 行，将损失 {cost} 行（{cost / len(df) * 100:.0f}%）')


> **决策原则**：损失 < 5% 可以考虑删；否则优先填充或删列。


---
# 二、重复值处理


In [ ]:
dup = pd.DataFrame({
    '订单号': [1001, 1002, 1003, 1004, 1005, 1006],
    '客户': ['张三', '张三', '李四', '王五', '李四', '张三'],
    '商品': ['A', 'A', 'B', 'C', 'B', 'D'],
    '金额': [100, 100, 200, 300, 200, 400],
})
print(dup)


In [ ]:
print('duplicated() 标记完全重复的行：')
print(dup.duplicated())

print('\n查看具体哪些行是重复的：')
print(dup[dup.duplicated()])


In [ ]:
print('drop_duplicates() 去重：')
print(dup.drop_duplicates())


### 只按某几列判断重复：`subset`（非常常用）


In [ ]:
print('按【客户+商品】去重（同一客户买同一商品只算一次）：')
print(dup.drop_duplicates(subset=['客户', '商品']))

print('\n按【客户】去重（每个客户只保留第一单）：')
print(dup.drop_duplicates(subset=['客户'], keep='first'))

print('\nkeep="last" 保留最后一单：')
print(dup.drop_duplicates(subset=['客户'], keep='last'))


**`drop_duplicates` 参数表**：

| 参数 | 含义 | 默认 |
|---|---|---|
| `subset` | 按哪几列判断重复 | 全部列 |
| `keep` | `'first'` / `'last'` / `False`（全删） | `'first'` |
| `ignore_index` | 是否重置索引 | `False` |

> **建议**：去重后接 `.reset_index(drop=True)`，
> 否则索引是断的（0,1,3,5...）。


In [ ]:
print('去重 + 重置索引：')
print(dup.drop_duplicates(subset=['客户'], ignore_index=True))


---
# 三、条件筛选：`query`

`query` 让你用**字符串**写筛选条件，比一堆方括号更接近自然语言。

先造一份有真实业务含义的数据：


In [ ]:
emp = pd.DataFrame({
    '姓名': ['张三', '李四', '王五', '赵六', '钱七'],
    '月薪': [15000, 8000, 25000, 18000, 12000],
    '部门': ['技术', '市场', '技术', '技术', '市场'],
    '性别': ['男', '女', '男', '男', '女'],
})
print(emp)


## 3.1 数值条件


In [ ]:
print("月薪 > 12000：")
print(emp.query('月薪 > 12000'))

print('\n月薪在 12000 ~ 20000 之间：')
print(emp.query('12000 <= 月薪 <= 20000'))


## 3.2 字符串条件（注意引号）


In [ ]:
print("部门 == '技术'：")
print(emp.query("部门 == '技术'"))
print('  <- 外层用双引号，里面用单引号；反过来也行，关键是别用同一种')


## 3.3 组合条件：`and` / `or` / `not`

**这是 `query` 比布尔索引更好用的地方** —— 不用写一堆括号！


In [ ]:
print("技术部 且 月薪 > 15000：")
print(emp.query("部门 == '技术' and 月薪 > 15000"))

print("\n技术部 或 月薪 > 20000（用 | 表示或）：")
print(emp.query("部门 == '技术' | 月薪 > 20000"))

print("\n取反（用 ~ 或 not）：")
print(emp.query("~(部门 == '技术')"))
print('等价写法：')
print(emp.query("not (部门 == '技术')"))


> **对比一下布尔索引的写法**：
>
> ```python
> # 布尔索引：括号不能少，符号容易记混
> emp[(emp['部门'] == '技术') & (emp['月薪'] > 15000)]
>
> # query：接近自然语言
> emp.query("部门 == '技术' and 月薪 > 15000")
> ```
>
> **`query` 更短更清楚**，列名还能直接写（不用 `emp['月薪']`）。
> **但它的坑是"列名有空格/特殊字符时要加反引号"**，见下。


## 3.4 用 `@` 引入 Python 变量（很实用）


In [ ]:
threshold = 15000
dept = '技术'

print(f'用 @ 引入外部变量（threshold={threshold}, dept={dept}）：')
print(emp.query('月薪 > @threshold and 部门 == @dept'))


**`@变量名`** 让查询条件可以动态变化 ——
写函数、做交互式报表时必备。


## 3.5 列名含空格时，用反引号包起来


In [ ]:
weird = pd.DataFrame({'员工 姓名': ['张三', '李四'], '月 薪': [100, 200]})
try:
    weird.query("月 薪 > 150")
except Exception as e:
    print('列名有空格，直接写 -> 报错：')
    print('  ', str(e)[:90])

print('\n用反引号包住列名：')
print(weird.query('`月 薪` > 150'))


---
# 四、按时间筛选

## 4.1 准备时间数据


In [ ]:
df_time = pd.DataFrame({
    'date': pd.date_range('2023-01-01', periods=100, freq='D'),
    'value': range(100),
})
print('数据范围：', df_time['date'].min().date(), '->', df_time['date'].max().date())
print(df_time.head(3))


## 4.2 三种筛选写法


In [ ]:
# 方法 1：布尔条件
r1 = df_time[(df_time['date'] >= '2023-01-15') & (df_time['date'] <= '2023-02-15')]
print('方法 1（布尔条件）：', len(r1), '条')

# 方法 2：between（更简洁）
r2 = df_time[df_time['date'].between('2023-01-15', '2023-02-15')]
print('方法 2（between）  ：', len(r2), '条')

# 方法 3：query
r3 = df_time.query("date >= '2023-01-15' and date <= '2023-02-15'")
print('方法 3（query）     ：', len(r3), '条')

print('\n三种方法结果一致吗？', len(r1) == len(r2) == len(r3))


## 4.3 设为索引后，用字符串直接切片（最简洁）


In [ ]:
di = df_time.set_index('date')

print("di.loc['2023-01-15':'2023-02-15']：")
print(len(di.loc['2023-01-15':'2023-02-15']), '条')

print("\n用 truncate 也行（before/after）：")
print(len(di.truncate(before='2023-01-15', after='2023-02-15')), '条')


## 4.4 按年 / 月 / 季度 / 周几筛选


In [ ]:
print('按年份：', len(df_time[df_time['date'].dt.year == 2023]), '条')
print('按月份：', len(df_time[df_time['date'].dt.month == 1]), '条')
print('按季度：', len(df_time[df_time['date'].dt.quarter == 1]), '条')

weekend = df_time[df_time['date'].dt.dayofweek >= 5]
print('周末：', len(weekend), '条（共 100 天，约 2/7 是周末，符合预期）')


## 4.5 按"几点"筛选：`between_time`

这需要**时间序列索引**（DatetimeIndex）。


In [ ]:
hourly = pd.DataFrame(
    {'销售额': np.arange(24) * 10},
    index=pd.date_range('2023-01-01', periods=24, freq='h'),
)
print('24 小时的销售数据：')
print(hourly.head(3))

# 营业时间 9:00 ~ 18:00
business = hourly.between_time('09:00', '18:00')
print('\n营业时间（9:00-18:00）共', len(business), '小时')
print('营业时间销售额合计：', business['销售额'].sum())
print('非营业时间销售额合计：', hourly['销售额'].sum() - business['销售额'].sum())


## 4.6 按时间重采样（顺带复习）

> ⚠️ **pandas 3.0 提醒**：月末用 `'ME'`、季末用 `'QE'`、年末用 `'YE'`，
> 老的 `'M'` / `'Q'` / `'Y'` 会**直接报错**。


In [ ]:
monthly = df_time.set_index('date').resample('ME')['value'].sum()
print('按月汇总：')
print(monthly)


### ⚠️ 时区坑：naive 和 aware 不能比


In [ ]:
tzdf = pd.DataFrame({'date': pd.to_datetime(['2023-08-15 10:00', '2023-08-16 10:00'])})
tzdf['utc'] = tzdf['date'].dt.tz_localize('UTC')

try:
    # 左边带时区，右边不带 -> 报错
    tzdf[tzdf['utc'] > pd.Timestamp.now()]
except TypeError as e:
    print('带时区和无时区的时间直接比较 -> 报错：')
    print('  ', str(e)[:120])

print('\n正确做法：两边都带时区')
safe = tzdf[tzdf['utc'] < pd.Timestamp.now(tz='UTC')]
print('筛选成功：', len(safe), '条')


---
# 五、字段分列（一列拆成多列）


In [ ]:
df_addr = pd.DataFrame({
    '地址': ['北京市 朝阳区 建国路', '上海市 浦东新区 陆家嘴', '广州市 天河区 珠江新城'],
    '邮编': ['100022', '200120', '510623'],
})
print(df_addr)


In [ ]:
# expand=True 是关键：让它返回 DataFrame 而不是 Series
df_addr[['省', '市区', '街道']] = df_addr['地址'].str.split(' ', expand=True)
print('按空格拆成 3 列：')
print(df_addr)


### ⚠️ `expand=True` 不能忘


In [ ]:
s = pd.Series(['a,b,c'])
print('不加 expand=True -> 返回 Series（元素是列表）：')
print(s.str.split(','))
print('类型：', type(s.str.split(',')).__name__)

print('\n加 expand=True -> 返回 DataFrame：')
print(s.str.split(',', expand=True))
print('类型：', type(s.str.split(',', expand=True)).__name__)


### 只保留需要的列：拆分后再挑


In [ ]:
df_info = pd.DataFrame({'信息': ['张三,25,男,北京', '李四,30,女,上海']})
print('原始：')
print(df_info)

# 拆出 4 列，但只要第 0 列（姓名）和第 2 列（性别）
df_info[['姓名', '性别']] = df_info['信息'].str.split(',', expand=True)[[0, 2]]
print('\n只取需要的两列：')
print(df_info)


### 拆分后出现缺失：`fillna` 补上


In [ ]:
df_part = pd.DataFrame({'姓名-年龄': ['张三,25', '李四', '王五,28']})
print('原始（李四没有年龄）：')
print(df_part)

df_part[['姓名', '年龄']] = df_part['姓名-年龄'].str.split(',', expand=True)
print('\n拆分后：')
print(df_part)
print('  <- 李四的年龄变成了 NaN（因为他没有逗号可拆）')

print('\n填充缺失：')
df_part['年龄'] = df_part['年龄'].fillna('未知')
print(df_part[['姓名', '年龄']])


### 分隔符不固定：用正则 `\s+`


In [ ]:
df_messy = pd.DataFrame({
    '信息': ['张三  25  男', '李四   30  女', '王五  28  男'],   # 空格数量不一致
})
print('原始（空格数不固定）：')
print(df_messy)

# regex=True 启用正则，\s+ 匹配"一个或多个空白"
df_messy[['姓名', '年龄', '性别']] = df_messy['信息'].str.split(r'\s+', expand=True, regex=True)
print('\n用正则拆分后：')
print(df_messy)


### 只拆前 N 个：`n` 参数


In [ ]:
df_long = pd.DataFrame({
    '地址': ['北京市 朝阳区 建国路 88号', '上海市 浦东新区 陆家嘴 1号'],
})
print('原始：')
print(df_long)

# n=1：只拆第 1 个空格，剩下的全归到第二列
df_long[['城市', '详细地址']] = df_long['地址'].str.split(' ', expand=True, n=1)
print('\nn=1 只拆一次（省/其余合成一列）：')
print(df_long)


---
# 六、字段合并（多列拼成一列）


In [ ]:
df_cat = pd.DataFrame({
    '省': ['广东', '浙江'],
    '市': ['深圳', '杭州'],
    '区': ['南山区', '西湖区'],
})
print(df_cat)


In [ ]:
# 方式 1：直接用 + 拼接（要注意自己加分隔符）
df_cat['地址1'] = df_cat['省'] + '省' + df_cat['市'] + '市' + df_cat['区']
print('方式 1（+ 号拼接）：')
print(df_cat[['地址1']])


In [ ]:
# 方式 2：str.cat 指定分隔符（更简洁）
df_cat['地址2'] = df_cat['省'].str.cat(df_cat['市'], sep='-').str.cat(df_cat['区'], sep='-')
print('方式 2（str.cat）：')
print(df_cat[['地址2']])


**两种方式对比**：

| 方式 | 优点 | 缺点 |
|---|---|---|
| `+` | 直观、能插入自定义字（如"省""市"） | 拼接项多了会很啰嗦 |
| `str.cat` | 统一分隔符，简洁 | 只能统一加分隔符 |
| `str.cat(others=[...])` | 一次拼多列 | — |


In [ ]:
# str.cat 一次拼多列的写法
df_cat['地址3'] = df_cat['省'].str.cat([df_cat['市'], df_cat['区']], sep='/')
print('一次拼多列：')
print(df_cat[['地址3']])


---
# 七、内容替换：`str.replace`


In [ ]:
s = pd.Series(['张三,25', '李四,30', '王五,28', '赵六,35'])
print('原始：')
print(s)

print('\n把逗号换成顿号：')
print(s.str.replace(',', '、', regex=False))


> **`regex=False` 是什么意思？**
> `str.replace` **默认把第一个参数当作正则表达式**。
> 逗号不是特殊字符，所以这里加不加 `regex=False` 结果一样，
> **但习惯性加上更安全**。
>
> 反过来：如果你**真的想用正则**，就别加 `regex=False`。


In [ ]:
print('用正则批量处理：把所有数字替换成 "#"：')
print(s.str.replace(r'\d+', '#', regex=True))


In [ ]:
print('多个值一起替换（传字典）：')
df_sex = pd.DataFrame({'性别': ['M', 'F', 'M', 'F']})
print(df_sex['性别'].replace({'M': '男', 'F': '女'}))


> **注意区分**：
> - **`Series.str.replace`** -> 处理**字符串内容**（支持正则）
> - **`Series/DataFrame.replace`** -> 处理**整个值**（精确替换、支持字典）
>
> 简单记：**替换"一部分"用 `.str.replace`，替换"整个值"用 `.replace`。**


---
# 八、信息提取：`str.extract`

场景：从"张三25岁"里**抠出数字 25**。


In [ ]:
s_age = pd.Series(['张三25岁', '李四30岁', '王五28岁', '赵六（35岁）', '孙七'])
print('原始：')
print(s_age)


**注意原稿的问题**：原稿写的是
```python
s.str.extract(r'正则表达式', expand=False)     # ❌ 这是占位符，跑不了
```
这里给出**真正能用的正则**。


In [ ]:
# \d+ 匹配 1 个或多个数字，用 () 捕获
age = s_age.str.extract(r'(\d+)', expand=False)
print('用 r"(\\d+)" 提取数字：')
print(age)

print('\n没有数字的"孙七"提取到的是 NaN（正常）')


In [ ]:
# 提取出来是字符串，记得转数值
print('转成数值类型：')
age_num = age.astype(float)     # 有 NaN 时用 float
print(age_num)


**`expand` 参数的区别**：

| 写法 | 返回 |
|---|---|
| `.str.extract(r'(\d+)', expand=False)` | **Series**（单个分组时推荐） |
| `.str.extract(r'(\d+)')` | **DataFrame**（列名是 0） |


In [ ]:
print('不加 expand=False 时返回 DataFrame：')
print(s_age.str.extract(r'(\d+)'))
print('  <- 列名是 0，因为只有一个捕获组')


### 提取多个字段（多个捕获组）


In [ ]:
s_full = pd.Series(['张三-25-男', '李四-30-女'])
print('原始：', s_full.tolist())

# 三个捕获组 -> 自动拆成三列
extracted = s_full.str.extract(r'(\w+)-(\d+)-(\w+)')
extracted.columns = ['姓名', '年龄', '性别']
print('\n提取结果：')
print(extracted)


---
# 九、一行拆多行：`explode`

场景：**一个格子里塞了一个列表**（比如"标签""关键词"）。


In [ ]:
df_tags = pd.DataFrame({
    '姓名': ['张三', '李四', '王五'],
    '标签': [['Python', '数据分析'], ['Java', '后端'], ['Excel', 'PPT', '办公']],
    '城市': ['北京', '上海', '广州'],
})
print('原始（标签是一整个列表）：')
print(df_tags)
print('形状：', df_tags.shape)


In [ ]:
exploded = df_tags.explode('标签', ignore_index=True)
print('explode 之后（每个标签占一行）：')
print(exploded)
print('形状：', exploded.shape)


**`explode` 的规则**：

| 原始行 | 标签数量 | 拆成几行 |
|---|---|---|
| 张三 | 2 | 2 |
| 李四 | 2 | 2 |
| 王五 | 3 | 3 |
| 合计 | 7 | **7 行** |

**其他列的值会被"复制"到每一行。**


In [ ]:
print('统计每个标签出现的次数：')
print(exploded['标签'].value_counts())


## 9.1 处理空列表和 NaN


In [ ]:
df_empty = pd.DataFrame({
    '姓名': ['张三', '李四', '王五'],
    '标签': [['Python', '数据分析'], [], None],     # 空列表 / None
})
print('原始：')
print(df_empty)

ex = df_empty.explode('标签', ignore_index=True)
print('\nexplode 之后（空列表会变成 NaN）：')
print(ex)

print('\n再用 dropna 清掉：')
print(ex.dropna(subset=['标签']))


## 9.2 同时拆多列（长度必须对应）


In [ ]:
df_multi = pd.DataFrame({
    '姓名': ['张三', '李四'],
    '标签': [['Python', '数据分析'], ['Java', '后端']],
    '分数': [[90, 85], [88, 92]],      # 和标签一一对应
})
print('原始：')
print(df_multi)

print('\n同时 explode 两列（按位置一一对应）：')
print(df_multi.explode(['标签', '分数'], ignore_index=True))


> **注意**：多列一起 explode 时，这几列的**列表长度必须一致**，
> 否则会报错。它们的元素会"按位置配对"到同一行。


## 9.3 经典组合：先分列 → 再 explode

字符串形式的"逗号分隔标签"要先拆成列表，才能 explode。

> 顺带把这两个操作的区别钉死 —— 它们长得像，用途完全不同：
>
> ![explode 与 分列 的区别](image.png)
>
> - **分列**：一格里是多个**不同类**的值（姓名+年龄），拆成**多列**
> - **explode**：一格里是多个**同类**的值（多个标签），炸成**多行**


In [ ]:
df_hobby = pd.DataFrame({
    '姓名': ['张三', '李四', '王五'],
    '爱好': ['篮球,足球,游泳', '读书,旅行', '跑步'],
})
print('原始（字符串形式）：')
print(df_hobby)

# 步骤 1：把字符串拆成列表
df_hobby['爱好列表'] = df_hobby['爱好'].str.split(',')
print('\n步骤 1：拆成列表')
print(df_hobby[['姓名', '爱好列表']])

# 步骤 2：explode
result = df_hobby.explode('爱好列表', ignore_index=True).drop(columns=['爱好'])
result = result.rename(columns={'爱好列表': '爱好'})
print('\n步骤 2：explode 之后')
print(result)


**为什么不能直接 explode 字符串？**

```python
df.explode('爱好')          # ❌ 字符串不是"可迭代的列表"，explode 不动它
```

`explode` 只认**列表 / Series / 数组**这类可迭代对象，
**字符串会被当成一个整体**。所以必须先 `str.split` 变成列表。


In [ ]:
print('\n直接 explode 字符串列，什么都不会发生：')
print(df_hobby.explode('爱好')[['姓名', '爱好']].head(3))
print('  <- 行数没变，因为"篮球,足球,游泳"被当成一个整体')


---
# 十、多行并成一行（文本聚合）

场景：**同一个人的多条记录，想把某个字段合并成一行**。

这是 expode 的**逆操作**。


In [ ]:
df_multi_row = pd.DataFrame({
    '姓名': ['张三', '张三', '张三', '李四', '李四', '王五'],
    '爱好': ['篮球', '足球', '游泳', '读书', '旅行', '跑步'],
    '城市': ['北京', '北京', '北京', '上海', '上海', '广州'],
})
print('原始（一个多行）：')
print(df_multi_row)
print('形状：', df_multi_row.shape)


In [ ]:
# 按姓名分组，把爱好用逗号连起来
merged = df_multi_row.groupby('姓名')['爱好'].agg(lambda x: ','.join(x)).reset_index()
print('合并后：')
print(merged)
print('形状：', merged.shape)


**关键在 `agg(lambda x: ','.join(x))`**：

- `groupby('姓名')` 把同一个人的多行分到一组
- `agg(函数)` 对每组的"爱好"列做聚合
- `lambda x: ','.join(x)` 把这一组的多个爱好用逗号连成一个字符串


In [ ]:
# 顺便把"城市"也保留下来（分组内相同，取第一个就行）
merged2 = df_multi_row.groupby('姓名').agg({
    '爱好': lambda x: ','.join(x),
    '城市': 'first',           # 也可以用 'last'
}).reset_index()
print('合并爱好 + 保留城市：')
print(merged2)


> **`'first'` / `'last'`** 是 pandas 内置的聚合函数，
> 分别取"分组内的第一个/最后一个值"。
> 当某列在组内**本来就相同**（比如城市）时，用 `'first'` 最省事。


In [ ]:
# 用不同分隔符
merged3 = df_multi_row.groupby('姓名')['爱好'].agg(' / '.join).reset_index()
print('用 " / " 分隔：')
print(merged3)


### `agg` 的三种传函数方式

| 写法 | 说明 |
|---|---|
| `agg(lambda x: ','.join(x))` | 匿名函数 |
| `agg(','.join)` | **直接传内置方法（更简洁）** |
| `agg(my_func)` | 自定义具名函数 |


In [ ]:
print("agg(','.join) 效果一样：")
print(df_multi_row.groupby('姓名')['爱好'].agg(','.join).reset_index())


---
## 常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| `mode()` 忘了 `[0]` | 填充报错 | `mode()[0]` |
| `fillna(method='ffill')` | pandas 3.0 报 `TypeError` | 用 `.ffill()` |
| 有极端值的列用均值填充 | 整体被带偏 | **用中位数** |
| 随手 `dropna()` | 样本大量损失 | 先算代价 |
| `query` 里字符串用双引号 | 语法错误 | 外双内单，或外单内双 |
| `query` 列名有空格 | `UndefinedVariableError` | 用**反引号**包住列名 |
| `str.split` 忘 `expand=True` | 赋值多列失败 | 拆多列必须加 |
| 多空格分隔用 `' '` | 拆出很多空列 | 用 `r'\s+'` + `regex=True` |
| 直接 `explode` 字符串列 | 毫无变化 | 先 `str.split` 成列表 |
| `str.extract` 写 `r'正则表达式'` | 提取不到东西 | 写真实正则，数字用 `(\d+)` |
| `str.replace` 想替换整个值 | 只换了子串 | 整个值替换用 `.replace` |
| naive 和 aware 时间比较 | `TypeError` | 两边都有时区 |
| `resample('M')` | pandas 3.0 报错 | 用 `'ME'` |

---

## 动手练习

**练习 1**：给这组数据分别用"均值""中位数""前向填充"处理，并说明各自适用场景：
`pd.DataFrame({'x': [10, np.nan, 30, np.nan, 1000]})`（注意有 1000 这个极端值！）

**练习 2**：把 `['北京的 朝阳区', '上海的 浦东新区']` 拆成"城市"和"区"两列。

**练习 3**：从 `['订单A-2023-01', '订单B-2023-02']` 里提取出年份和月份。

**练习 4**：把这份数据从"列表形式"拆成多行：
`{'用户': ['u1','u2'], '权限': [['读','写'], ['读']]}`

**练习 5**：把上题的结果"再合并回去"（每个用户的权限用 `|` 连接成一行）。

**练习 6**：有一份员工数据，用 `query` 找出"月薪大于等于 15000 的男性技术员工"。

**练习 7**：给一份 2023 年每天的销售数据，筛选出"第二季度（4-6月）且在工作日"的记录。

---

**答案思路（先自己做）**

```python
# 1
d = pd.DataFrame({'x': [10, np.nan, 30, np.nan, 1000]})
print('均值填充：', d['x'].fillna(d['x'].mean()).tolist())      # 被 1000 带偏
print('中位数填充：', d['x'].fillna(d['x'].median()).tolist())  # 推荐
print('前向填充：', d['x'].ffill().tolist())

# 2
a = pd.DataFrame({'地址': ['北京的 朝阳区', '上海的 浦东新区']})
a[['城市', '区']] = a['地址'].str.split(' ', expand=True)

# 3
o = pd.Series(['订单A-2023-01', '订单B-2023-02'])
print(o.str.extract(r'-(\d{4})-(\d{2})'))

# 4
u = pd.DataFrame({'用户': ['u1', 'u2'], '权限': [['读', '写'], ['读']]})
print(u.explode('权限', ignore_index=True))

# 5
e = u.explode('权限', ignore_index=True)
print(e.groupby('用户')['权限'].agg('|'.join).reset_index())

# 6
emp.query("月薪 >= 15000 and 性别 == '男' and 部门 == '技术'")

# 7
s = df_time.set_index('date')
q2 = s[(s.index.month.isin([4, 5, 6])) & (s.index.dayofweek < 5)]
```
